In [22]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

In [38]:
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "microsoft/phi-2"  # 2.7B, fully open, no gating

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    device_map="auto",
    torch_dtype=torch.float16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
# target_modules = ['q_proj', 'k_proj', 'v_proj']

# for i, layer in enumerate(model.model.layers):
#     for name in target_modules:
#         original = getattr(layer.self_attn, name)
#         setattr(layer.self_attn, name, LoRALinear(original))

# # Verify
# total = sum(p.numel() for p in model.parameters())
# trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
# print(f"Trainable: {trainable:,} / {total:,}")

In [39]:
import torch.nn as nn

In [40]:
class LoRALinear(nn.Module):
    def __init__(self, linear, rank=512 ):
        super().__init__()
        self.linear = linear
        self.linear.weight.requires_grad = False
        
        in_features = linear.weight.shape[1]
        out_features = linear.weight.shape[0]
        
        self.A = nn.Linear(in_features, rank, bias = False)
        self.B = nn.Linear(rank, out_features, bias = False)
        self.A.weight.requires_grad = True
        self.B.weight.requires_grad = True
        
        nn.init.normal_(self.A.weight)
        nn.init.zeros_(self.B.weight)
        
    def forward(self, x):
        return self.linear(x) + self.B(self.A(x))
    

In [41]:
for param in model.parameters():
    param.requires_grad = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} / {total:,}")

Trainable: 0 / 2,779,683,840


In [42]:
target_modules = ['q_proj', 'k_proj', 'v_proj']

for layer in model.model.layers:
    for name in target_modules:
        original = getattr(layer.self_attn, name)
        if isinstance(original, LoRALinear):
            continue  # already injected, skip
        setattr(layer.self_attn, name, LoRALinear(original))

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} / {total:,}")

Trainable: 251,658,240 / 3,031,342,080


In [43]:
# for param in model.parameters():
#     param.requires_grad = False

total = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable: {trainable:,} / {total:,}")

Trainable: 251,658,240 / 3,031,342,080
